# Model Evaluation

This notebook evaluates the selected final model from Phase 7.

The evaluation focuses on classification performance using the held-out test dataset. No model retraining is performed in this notebook.

## 8.1 Classification Evaluation

The selected Logistic Regression model is evaluated using standard classification metrics:

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Classification Report

In [1]:
import sys
from pathlib import Path

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

PROJECT_ROOT_81 = Path.cwd().parent

if str(PROJECT_ROOT_81) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_81))

from backend.src.ml.preprocessing_pipeline import build_preprocessing_pipeline

In [2]:
import joblib

MODEL_PATH_81 = (
    PROJECT_ROOT_81
    / "backend"
    / "artifacts"
    / "models"
    / "logistic_regression_final.joblib"
)

final_model_81 = joblib.load(MODEL_PATH_81)

print("Final model loaded successfully.")
print("Model type:", type(final_model_81).__name__)

Final model loaded successfully.
Model type: Pipeline


In [3]:
from sklearn.model_selection import train_test_split

DATA_PATH_81 = (
    PROJECT_ROOT_81
    / "data"
    / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df_81 = pd.read_csv(DATA_PATH_81)

X_81 = df_81.drop(columns=["customerID", "Churn"])

y_81 = df_81["Churn"].map({
    "No": 0,
    "Yes": 1
})

X_train_81, X_test_81, y_train_81, y_test_81 = train_test_split(
    X_81,
    y_81,
    test_size=0.20,
    random_state=42,
    stratify=y_81
)

print("Evaluation dataset prepared.")
print("Test shape:", X_test_81.shape)
print("Test target shape:", y_test_81.shape)

Evaluation dataset prepared.
Test shape: (1409, 19)
Test target shape: (1409,)


In [4]:
y_pred_81 = final_model_81.predict(X_test_81)
y_prob_81 = final_model_81.predict_proba(X_test_81)[:, 1]

metrics_81 = {
    "Accuracy": accuracy_score(y_test_81, y_pred_81),
    "Precision": precision_score(y_test_81, y_pred_81),
    "Recall": recall_score(y_test_81, y_pred_81),
    "F1 Score": f1_score(y_test_81, y_pred_81),
    "ROC-AUC": roc_auc_score(y_test_81, y_prob_81)
}

print("=" * 60)
print("FINAL MODEL CLASSIFICATION METRICS")
print("=" * 60)

for metric, score in metrics_81.items():
    print(f"{metric:<10}: {score:.4f}")

FINAL MODEL CLASSIFICATION METRICS
Accuracy  : 0.7999
Precision : 0.6554
Recall    : 0.5187
F1 Score  : 0.5791
ROC-AUC   : 0.8424


In [5]:
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test_81,
        y_pred_81,
        target_names=["No Churn", "Churn"]
    )
)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

    No Churn       0.84      0.90      0.87      1035
       Churn       0.66      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

